# notebook_person_mention_schema_init v1.0 — SUPERSEDED, historical reference only

**Status as of ocr_pipeline.ipynb v4:** this notebook's job — owning and creating
`genealogy.silver_transcript_person_mention` — has moved to `ocr_pipeline.ipynb`
(its idempotent `CREATE TABLE IF NOT EXISTS` + `ALTER TABLE ... ADD COLUMN` loop,
same pattern already used there for `ocr_transcriptions`). Do not re-run Cell 1
below — its `CREATE OR REPLACE TABLE` would destroy the real data the table now
holds, and `ocr_pipeline.ipynb` is the schema of record going forward. Kept here
as the historical record of the table's original v1.0/v1.1 design.

**Run order (historical):** Standalone — no upstream dependency. Run once to create the table; safe to re-run (idempotent `CREATE OR REPLACE TABLE`, but note this drops and recreates the table, so never re-run after it holds real data).

**Purpose (historical — see status note above for the current pipeline):** Creates `genealogy.silver_transcript_person_mention`, a tidy/long-format table holding structured per-person records extracted from `ocr_transcriptions.transcribed_text` via Claude — not Gemini/OCR, since the raw transcript is already captured. Rows were populated live in chat/Cowork sessions (Ed reads transcripts via the Databricks MCP, structures them with Claude, writes results back) during the Sept 2026 backfill (Asana 1218641074662915). As of `ocr_pipeline.ipynb` v4, Gemini populates this table directly at transcription time instead — see that notebook's `ROLE_GUIDANCE` and `build_prompt` for the current extraction logic, and its schema cell for the table's current definition (adds `page_index`, `corrected_at`, `correction_notes` beyond what's defined below).

Covers any document with named people in it, not just household-style records — a will's executor/legatees/witnesses are structurally the same shape (repeating people, each with a role) as a census household. `role_in_record` uses a controlled vocabulary spanning both: household roles (Head, Wife, Son, Daughter, Boarder, Visitor), baptism/burial roles (Father, Mother, Child, Deceased, Informant), and document roles (Executor, Legatee, Witness). Content specific to a role that isn't age/DOB — a bequest's description, a witness's stated occupation — goes in `detail`; `notes` stays reserved for extraction-quality flags only (ditto marks, unclear digits), never content. Single-value document-level facts with no person attached (date of will, probate date) belong in the separate, narrower `silver_transcript_document_field` (not this notebook).

One row per person mentioned in a document; deliberately has no `person_gedcom_id` — matching against tree candidates stays `notebook_01_document_matching.ipynb` Cell 5e's job, just working from these structured rows instead of reverse-engineering flat OCR text.

See Asana task 1218641074662915 (prerequisite for 1218420653379642, household-member document matching) and design discussion on 1218429902053332.

**Latest changes:** Renamed `relation_to_head` to `role_in_record` and added `detail` (v1.1) to support non-household documents (wills: executor/legatee/witness) with the same table, after reviewing Karen Cummings' will-extraction structure alongside her census structure.

In [ ]:
%sql
-- Delta TABLE (not view): populated incrementally, row-by-row, across many
-- chat sessions rather than recomputed from a query — a view has nothing to
-- select from here.
CREATE OR REPLACE TABLE genealogy.silver_transcript_person_mention (
  file_id STRING COMMENT 'Google Drive file ID — foreign key to ocr_transcriptions',
  person_index INT COMMENT 'Order of this person within the document (0-based)',
  name_raw STRING COMMENT 'Name as transcribed, unnormalised',
  role_in_record STRING COMMENT 'This person''s role as stated in the document — controlled vocabulary spanning household roles (Head, Wife, Son, Daughter, Boarder, Visitor), baptism/burial roles (Father, Mother, Child, Deceased, Informant), and document roles (Executor, Legatee, Witness). Shared vocabulary with the future Gemini extraction prompt (1218421366058506) so Cell 5e sees one consistent set regardless of which model wrote the row. Null where not applicable.',
  age_raw STRING COMMENT 'Age as transcribed, preserving original text (e.g. "abt 40", "6 mo")',
  age_years INT COMMENT 'Parsed numeric age in years, null if not resolvable',
  dob_raw STRING COMMENT 'Date of birth as transcribed, if present',
  detail STRING COMMENT 'Role-specific content that is not age/DOB, e.g. a bequest''s description for a Legatee row, a witness''s stated occupation. Content, not a quality flag — see notes.',
  notes STRING COMMENT 'Extraction-quality flags only — e.g. ditto marks, unclear digits, conflicting entries. Never document content; content belongs in detail.',
  confidence STRING COMMENT 'high | medium | low — self-assessed during extraction',
  model_used STRING COMMENT 'Model that produced this row, e.g. claude-sonnet-4-6',
  extracted_at TIMESTAMP COMMENT 'When this row was written'
)
USING DELTA
COMMENT 'Structured per-person records extracted from ocr_transcriptions.transcribed_text via Claude (not Gemini/OCR) — one row per person mentioned in a document, not yet matched to a tree person. Feeds notebook_01 Cell 5e household-member matching. See Asana task 1218641074662915.'

In [ ]:
%sql
-- expect 0 rows immediately after creation
DESCRIBE TABLE genealogy.silver_transcript_person_mention;
SELECT COUNT(*) AS row_count FROM genealogy.silver_transcript_person_mention